# Props Data Organizer

## Definitions of Formulas Used in This Notebook

### Estimated Value (EV)
EV is the average amount you can expect to win or lose per bet if you placed the same bet many times. It helps identify profitable betting opportunities by comparing the expected return to the risk involved.

**Formula:**
$$
\text{EV} = (\text{Probability of Winning} \times \text{Profit if Win}) - (\text{Probability of Losing} \times \text{Loss if Lose})
$$

### Kelly Criterion
The Kelly Criterion is a formula used to determine the optimal size of a series of bets. It aims to maximize the logarithm of wealth, balancing the trade-off between risk and reward. The formula considers both the probability of winning and the odds offered, guiding you on how much of your bankroll to wager on each bet.

**Formula:**
$$
\text{Kelly Fraction} = \frac{(\text{Probability of Winning} \times (\text{Odds} + 1)) - 1}{\text{Odds}}
$$

### Variance
Variance in sports betting represents the spread or dispersion of actual outcomes around the expected value. It's a crucial metric for understanding the risk and volatility associated with betting predictions. Higher variance indicates more volatile and unpredictable outcomes, while lower variance suggests more consistent results.

**Formula:**
$$
\text{Variance} = \frac{\sum_{i=1}^{n} (x_i - \mu)^2}{n}
$$

Where:
- $x_i$ represents each individual outcome
- $\mu$ is the mean or expected value
- $n$ is the total number of observations

In the context of prop betting:
- High variance props (e.g., 3-pointers made) tend to be more risky but potentially more profitable
- Low variance props (e.g., minutes played) typically offer more consistent but lower returns


In [2]:
import pandas as pd 
import numpy as np
import time
import requests
import os
import sys
from datetime import datetime
import joblib

feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
# feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)
    
from PROPS_EV.calculateEVS import *
from MODELS.model import *

today = datetime.now()
formatted_date = today.strftime("%m_%d_%y")

c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Grabs players odds for the day (US all boookmakers, DFS is prizepicks and underdogs)

In [3]:
# from NBAPropFinder.NBAPropFinder import NBAPropFinder

# nba_props = NBAPropFinder(region='us_dfs')
# prizePicks = nba_props.dataframe
# prizePicks.head(10)

### Single Bets from bookmakers that dont include prizePicks or UnderDogs

In [ ]:
features = [
    # Player context
    'PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID', 
    'STARTING', 'HOME_GAME', 
    'PLAYER_DAYS_REST', 'IS_BACK_TO_BACK', 
    
    # Player season averages
    'PTS_AVG_TO_DATE', 'MIN_AVG_TO_DATE', 'FGA_AVG_TO_DATE', 'FTA_AVG_TO_DATE', 'FG3A_AVG_TO_DATE', 
    'FG_PCT_AVG_TO_DATE', 'FG3_PCT_AVG_TO_DATE', 'FT_PCT_AVG_TO_DATE', 'USG_PCT_AVG_TO_DATE', 'TS_PCT_AVG_TO_DATE', 
    'EFG_PCT_AVG_TO_DATE', 'POSS_AVG_TO_DATE', 'TCHS_AVG_TO_DATE', 'AST_AVG_TO_DATE', 'REB_AVG_TO_DATE', 'TOV_AVG_TO_DATE',
    
    # LAG
    'PTS_LAG_1', 'PTS_LAG_2', 'MIN_LAG_1', 'MIN_LAG_2', 'FGA_LAG_1', 'FGA_LAG_2', 'FTA_LAG_1', 'FTA_LAG_2', 'FG3A_LAG_1', 
    'FG3A_LAG_2', 'FG_PCT_LAG_1', 'FG_PCT_LAG_2', 'FG3_PCT_LAG_1', 'FG3_PCT_LAG_2', 'FT_PCT_LAG_1', 'FT_PCT_LAG_2', 
    'USG_PCT_LAG_1', 'USG_PCT_LAG_2', 'TS_PCT_LAG_1', 'TS_PCT_LAG_2', 'EFG_PCT_LAG_1', 'EFG_PCT_LAG_2', 'POSS_LAG_1', 'POSS_LAG_2', 
    'TCHS_LAG_1', 'TCHS_LAG_2', 'AST_LAG_1', 'AST_LAG_2', 'REB_LAG_1', 'REB_LAG_2', 'TOV_LAG_1', 'TOV_LAG_2',
    
    # 3 game rolling averages
    'PTS_ROLLING_AVG_3', 'MIN_ROLLING_AVG_3', 'FGA_ROLLING_AVG_3', 'FTA_ROLLING_AVG_3', 'FG3A_ROLLING_AVG_3', 'FG_PCT_ROLLING_AVG_3', 
    'FG3_PCT_ROLLING_AVG_3', 'FT_PCT_ROLLING_AVG_3', 'USG_PCT_ROLLING_AVG_3', 'TS_PCT_ROLLING_AVG_3', 'EFG_PCT_ROLLING_AVG_3', 
    'POSS_ROLLING_AVG_3', 'TCHS_ROLLING_AVG_3', 'AST_ROLLING_AVG_3', 'REB_ROLLING_AVG_3', 'TOV_ROLLING_AVG_3',

    # Short-term form (5-game rolling averages)
    'PTS_ROLLING_AVG_5', 'MIN_ROLLING_AVG_5', 'FGA_ROLLING_AVG_5', 'FTA_ROLLING_AVG_5', 'FG3A_ROLLING_AVG_5', 'FG_PCT_ROLLING_AVG_5', 
    'FG3_PCT_ROLLING_AVG_5', 'FT_PCT_ROLLING_AVG_5', 'USG_PCT_ROLLING_AVG_5', 'TS_PCT_ROLLING_AVG_5', 'EFG_PCT_ROLLING_AVG_5', 
    'POSS_ROLLING_AVG_5', 'TCHS_ROLLING_AVG_5', 'AST_ROLLING_AVG_5', 'REB_ROLLING_AVG_5', 'TOV_ROLLING_AVG_5',
    
    # 7 game rolling averages
    'PTS_ROLLING_AVG_7', 'MIN_ROLLING_AVG_7', 'FGA_ROLLING_AVG_7', 'FTA_ROLLING_AVG_7', 'FG3A_ROLLING_AVG_7', 'FG_PCT_ROLLING_AVG_7', 
    'FG3_PCT_ROLLING_AVG_7', 'FT_PCT_ROLLING_AVG_7', 'USG_PCT_ROLLING_AVG_7', 'TS_PCT_ROLLING_AVG_7', 'EFG_PCT_ROLLING_AVG_7', 
    'POSS_ROLLING_AVG_7', 'TCHS_ROLLING_AVG_7', 'AST_ROLLING_AVG_7', 'REB_ROLLING_AVG_7', 'TOV_ROLLING_AVG_7',

    # Medium-term form (15-game rolling averages)
    'PTS_ROLLING_AVG_15', 'MIN_ROLLING_AVG_15', 'FGA_ROLLING_AVG_15', 'FTA_ROLLING_AVG_15', 'FG3A_ROLLING_AVG_15', 'FG_PCT_ROLLING_AVG_15', 
    'FG3_PCT_ROLLING_AVG_15', 'FT_PCT_ROLLING_AVG_15', 'USG_PCT_ROLLING_AVG_15', 'TS_PCT_ROLLING_AVG_15', 'EFG_PCT_ROLLING_AVG_15', 
    'POSS_ROLLING_AVG_15', 'TCHS_ROLLING_AVG_15', 'AST_ROLLING_AVG_15', 'REB_ROLLING_AVG_15', 'TOV_ROLLING_AVG_15',
    
    # Long-term form (40-game rolling averages)
    'PTS_ROLLING_AVG_40', 'MIN_ROLLING_AVG_40', 'FGA_ROLLING_AVG_40', 'FTA_ROLLING_AVG_40', 'FG3A_ROLLING_AVG_40', 'FG_PCT_ROLLING_AVG_40', 
    'FG3_PCT_ROLLING_AVG_40', 'FT_PCT_ROLLING_AVG_40', 'USG_PCT_ROLLING_AVG_40', 'TS_PCT_ROLLING_AVG_40', 'EFG_PCT_ROLLING_AVG_40', 
    'POSS_ROLLING_AVG_40', 'TCHS_ROLLING_AVG_40', 'AST_ROLLING_AVG_40', 'REB_ROLLING_AVG_40', 'TOV_ROLLING_AVG_40',

    # Opponent
    'OPP_DEF_RATING_AVG_TO_DATE', 'OPP_PACE_AVG_TO_DATE', 'OPP_PTS_AVG_TO_DATE', 'OPP_FGA_AVG_TO_DATE', 
    'OPP_REB_AVG_TO_DATE', 'OPP_AST_AVG_TO_DATE', 'OPP_TOV_AVG_TO_DATE', 'OPP_BLK_AVG_TO_DATE', 'OPP_STL_AVG_TO_DATE',
    
    #starters 
    'TEAM_OFF_RATING_AVG_TO_DATE','TEAM_DEF_RATING_AVG_TO_DATE','TEAM_PACE_AVG_TO_DATE', 'TEAM_FGA_AVG_TO_DATE',
    'TEAM_PTS_AVG_TO_DATE', 'TEAM_REB_AVG_TO_DATE', 'TEAM_AST_AVG_TO_DATE', 'TEAM_TOV_AVG_TO_DATE',
    
    # Team odds
    'team_spread', 'total', 'team_is_favored','TEAM_IMPLIED_PTS_FAV','TEAM_IMPLIED_PTS_UND','BLOWOUT_RISK'
]

# model = joblib.load('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/Models/PTS_cat_model.pkl')
model = joblib.load(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS\Models\PTS_cat_model.pkl")
data = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values('GAME_DATE', ascending=False)
bookmakers = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
date = '2025-04-11'
espnDate = '20250411'
odds = bookmakers[
    (bookmakers['CATEGORY'] == 'points') &
    (bookmakers['GAME_DATE'] == date) &
    (bookmakers['ODDS'] < 200) &
    (bookmakers['ODDS'] > -200)
]
filterData = data[data['GAME_DATE'] <= date].sort_values('GAME_DATE', ascending=True)
games = get_espn_games(date_str=espnDate)
odds.sort_values(by='ODDS').head(25)

,NAME,CATEGORY,SIDE,BOOKMAKER,LINE,ODDS,fair_line,fair_odds,GAME_DATE
230605,Kristaps Porzingis,points,over,fanduel,19.5,-198,19.5,118,2025-04-11
229365,Trae Young,points,under,draftkings,37.5,-195,33.5,112,2025-04-11
232415,Ja Morant,points,under,fanduel,23.5,-192,27.5,-104,2025-04-11
229386,Dyson Daniels,points,over,espnbet,9.5,-190,8.5,-105,2025-04-11
231659,Kyle Anderson,points,under,espnbet,0.5,-190,0.5,-146,2025-04-11
230918,Mitchell Robinson,points,over,espnbet,1.5,-190,1.5,-154,2025-04-11
232109,Julius Randle,points,over,fanduel,19.5,-188,21.5,-104,2025-04-11
230434,Jalen Duren,points,under,fanduel,12.5,-186,13.5,106,2025-04-11
230971,Mikal Bridges,points,under,fanduel,17.5,-186,17.5,-148,2025-04-11
232342,Mike Conley,points,over,draftkings,7.5,-185,7.5,-105,2025-04-11


## Best EVs for Single Bets from draftkings, fanduel, prizepicks, and underdog

In [9]:
final_results = single_bet(filterData, odds, model, games, features, '20250411', stake=100, simulations=10000).sort_values(by='EV%', ascending=False).reset_index(drop=True)

print("\nTop 10 highest EV bets across point props:")
final_results.head(10)

Processing single bets...

DEBUG - Nickeil Alexander-Walker:
  Odds: -130 (over)
  Line: 10.5
  Prediction: 7.91
  Std Dev: 3.238655413730965
  Prob Over: 0.211
  Decimal Odds: 1.77
  Breakeven: 0.565
  Simulated Mean: 7.97
  Model Prediction: 7.91

DEBUG - Nickeil Alexander-Walker:
  Odds: -102 (under)
  Line: 10.5
  Prediction: 7.91
  Std Dev: 3.238655413730965
  Prob Over: 0.206
  Decimal Odds: 1.98
  Breakeven: 0.505
  Simulated Mean: 7.96
  Model Prediction: 7.91

DEBUG - Rudy Gobert:
  Odds: -115 (under)
  Line: 12.5
  Prediction: 13.09
  Std Dev: 5.486346689738082
  Prob Over: 0.548
  Decimal Odds: 1.87
  Breakeven: 0.535
  Simulated Mean: 13.27
  Model Prediction: 13.09

DEBUG - Rudy Gobert:
  Odds: -121 (under)
  Line: 12.5
  Prediction: 13.09
  Std Dev: 5.486346689738082
  Prob Over: 0.545
  Decimal Odds: 1.83
  Breakeven: 0.548
  Simulated Mean: 13.20
  Model Prediction: 13.09

DEBUG - Rudy Gobert:
  Odds: -115 (over)
  Line: 12.5
  Prediction: 13.09
  Std Dev: 5.48634668973

,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,OVER%,UNDER%,BREAKEVEN%,EDGE%,EV$,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
0,Derrick White,fanduel,points,3.5,132,over,15.28,0.997,0.003,43.1,56.6,131.30,131.30,0.99,0.50,0.25,"(6.9, 23.5)"
1,Anthony Black,espnbet,points,2.5,130,over,8.50,0.961,0.039,43.5,52.6,121.08,121.08,0.93,0.47,0.23,"(2.0, 15.6)"
2,Jrue Holiday,fanduel,points,3.5,130,over,11.32,0.945,0.055,43.5,51.0,117.28,117.28,0.90,0.45,0.23,"(2.1, 21.7)"
3,Aaron Nesmith,espnbet,points,2.5,105,over,13.08,1.000,0.000,48.8,51.2,104.92,104.92,1.00,0.50,0.25,"(6.7, 19.3)"
4,Darius Garland,espnbet,points,5.5,115,over,19.98,0.952,0.048,46.5,48.7,104.66,104.66,0.91,0.46,0.23,"(3.6, 38.6)"
5,Thomas Bryant,fanatics,points,14.5,105,under,8.28,0.008,0.992,48.8,50.4,103.30,103.30,0.98,0.49,0.25,"(3.2, 13.4)"
6,Quentin Grimes,espnbet,points,7.5,100,over,18.31,1.000,0.000,50.0,50.0,99.96,99.96,1.00,0.50,0.25,"(12.2, 24.4)"
7,Adem Bona,espnbet,points,17.5,100,under,10.42,0.000,1.000,50.0,50.0,99.92,99.92,1.00,0.50,0.25,"(6.5, 14.3)"
8,Anthony Black,espnbet,points,20.5,100,under,8.50,0.001,0.999,50.0,49.9,99.86,99.86,1.00,0.50,0.25,"(1.8, 15.5)"
9,Jayson Tatum,fanduel,points,8.5,100,over,26.21,0.997,0.003,50.0,49.7,99.42,99.42,0.99,0.50,0.25,"(13.1, 39.7)"


In [10]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import sys
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass
import joblib
from pathlib import Path
import warnings
import requests
import pytz
warnings.filterwarnings('ignore')

# Fixed get_espn_games function with correct date format
def get_espn_games(date_str):
    """
    Get ESPN games for a specific date
    Args:
        date_str: Date in YYYYMMDD format (e.g., '20241022')
    Returns:
        List of game dictionaries
    """
    try:
        # ESPN API expects YYYYMMDD format
        url = f"http://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?dates={date_str}"
        response = requests.get(url, timeout=10)
        
        if response.status_code != 200:
            return []
            
        data = response.json()
        
        if 'events' not in data:
            return []
        
        # Define timezone objects
        utc = pytz.UTC
        pst = pytz.timezone('America/Los_Angeles')

        games_list = []
        for event in data['events']:
            try:
                # Parse UTC time from ESPN
                utc_time = datetime.strptime(event['date'], '%Y-%m-%dT%H:%MZ').replace(tzinfo=utc)
                # Convert to PST
                pst_time = utc_time.astimezone(pst)
                
                # Get team abbreviations
                competitors = event['competitions'][0]['competitors']
                home_team = None
                away_team = None
                
                for competitor in competitors:
                    if competitor['homeAway'] == 'home':
                        home_team = competitor['team']['abbreviation']
                    else:
                        away_team = competitor['team']['abbreviation']
                
                game_dict = {
                    'game_date': pst_time.strftime('%Y-%m-%d'),
                    'home_team': home_team,
                    'away_team': away_team,
                    'game_time': pst_time.strftime('%I:%M %p'),
                    'venue': event['competitions'][0]['venue']['fullName'] if 'venue' in event['competitions'][0] else 'Unknown'
                }
                games_list.append(game_dict)
                
            except Exception as e:
                continue
        
        return games_list
        
    except Exception as e:
        return []

@dataclass
class BetResult:
    """Data class to store individual bet results"""
    date: str
    bet_type: str = 'single'
    player_name: str = ''
    player2_name: Optional[str] = None
    category: str = 'player_points'
    category2: Optional[str] = None
    line: float = 0.0
    line2: Optional[float] = None
    side: str = 'over'
    side2: Optional[str] = None
    prediction: float = 0.0
    prediction2: Optional[float] = None
    actual_result: float = 0.0
    actual_result2: Optional[float] = None
    probability: float = 0.5
    ev_percent: float = 0.0
    ev_dollars: float = 0.0
    kelly_fraction: float = 0.01
    stake_amount: float = 0.0
    won: bool = False
    profit_loss: float = 0.0
    bookmaker: str = 'prizepicks'

class CleanNBABacktester:
    """
    Clean backtest class with minimal output
    """
    
    def __init__(self, 
                 data_path: str = r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\DATA\CSV_FILES\TRAIN_DATA\PTS_TRAIN_25.csv",
                 odds_path: str = r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\BACKTESTS\singleBets.csv",
                 model_path: str = r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS\Models\PTS_cat_model.pkl",
                 min_ev_threshold: float = 0.02,
                 base_stake: float = 100,
                 kelly_multiplier: float = 0.20,
                 max_bets_per_day: int = 3,
                 start_date: str = None,
                 end_date: str = None):
        
        self.min_ev_threshold = min_ev_threshold
        self.base_stake = base_stake
        self.kelly_multiplier = kelly_multiplier
        self.max_bets_per_day = max_bets_per_day
        self.start_date = start_date
        self.end_date = end_date
        
        # Load data and model silently
        try:
            self.data = pd.read_csv(data_path, low_memory=False).sort_values('GAME_DATE', ascending=True)
            self.bookmakers = pd.read_csv(odds_path, low_memory=False)
            self.model = joblib.load(model_path)
        except Exception as e:
            raise Exception(f"Error loading data: {e}")
        
        # Results storage
        self.bet_results: List[BetResult] = []
        self.daily_summary: List[Dict] = []
        self.processing_log: List[Dict] = []
    
    def get_unique_dates(self) -> List[str]:
        """Get unique game dates from bookmakers data"""
        try:
            dates = self.bookmakers['GAME_DATE'].unique()
            dates = [d for d in dates if pd.notna(d)]
            dates = sorted(dates)
            
            if self.start_date:
                dates = [d for d in dates if d >= self.start_date]
            if self.end_date:
                dates = [d for d in dates if d <= self.end_date]
                
            return dates
        except Exception as e:
            return []
    
    def get_actual_result(self, player_name: str, game_date: str, stat_type: str = 'PTS') -> Optional[float]:
        """Get actual game result for a player on a specific date"""
        try:
            game_data = self.data[
                (self.data['PLAYER_NAME'] == player_name) & 
                (self.data['GAME_DATE'] == game_date)
            ]
            
            if game_data.empty or stat_type not in game_data.columns:
                return None
                
            return float(game_data[stat_type].iloc[0])
        except Exception as e:
            return None
    
    def enhanced_single_bet(self, game_date: str) -> pd.DataFrame:
        """Enhanced version with better game data handling"""
        try:
            # Get odds for the date
            single_bet_books = [
                'bovada', 'espnbet','fanduel',
                'betmgm', 'draftkings', 'caesars', 'betrivers',
                'pinnacle', 'bet365'
            ]

            daily_odds = self.bookmakers[
                (self.bookmakers['CATEGORY'] == 'points') &
                (self.bookmakers['GAME_DATE'] == game_date) &
                (self.bookmakers['BOOKMAKER'].isin(single_bet_books)) &
                (self.bookmakers['ODDS'] < 200) &
                (self.bookmakers['ODDS'] > -200)
            ].copy()

            if daily_odds.empty:
                return pd.DataFrame()
            
            # Convert date format for ESPN API
            date_obj = datetime.strptime(game_date, "%Y-%m-%d")
            date_str = date_obj.strftime("%Y%m%d")  # Convert to YYYYMMDD
            
            # Get games silently
            games = get_espn_games(date_str)
            if not games:
                games = []  # Proceed without games data
            
            # Get training data
            filtered_data = self.data[self.data['GAME_DATE'] < game_date].sort_values('GAME_DATE', ascending=True)
            
            if filtered_data.empty:
                return pd.DataFrame()
            
            # Enhanced prediction logic
            results = []
            
            for _, row in daily_odds.iterrows():
                try:
                    player_name = row['NAME']
                    line = float(row['LINE'])
                    side = row.get('SIDE', 'over')
                    
                    # Get player's recent performance
                    player_data = filtered_data[filtered_data['PLAYER_NAME'] == player_name]
                    
                    if player_data.empty:
                        continue
                    
                    # Enhanced prediction using multiple metrics
                    recent_5 = player_data['PTS'].tail(5).mean()
                    recent_10 = player_data['PTS'].tail(10).mean()
                    season_avg = player_data['PTS'].mean()
                    
                    # Weight recent games more heavily
                    prediction = (recent_5 * 0.5 + recent_10 * 0.3 + season_avg * 0.2)
                    
                    if pd.isna(prediction):
                        continue
                    
                    # Calculate standard deviation for better probability estimation
                    recent_std = player_data['PTS'].tail(10).std()
                    if pd.isna(recent_std) or recent_std == 0:
                        recent_std = 5.0  # Default std
                    
                    # Use normal distribution to estimate probability
                    from scipy import stats
                    if side.upper().startswith('O'):
                        prob = 1 - stats.norm.cdf(line, prediction, recent_std)
                    else:
                        prob = stats.norm.cdf(line, prediction, recent_std)
                    
                    # Ensure probability is reasonable
                    prob = max(0.1, min(0.9, prob))
                    
                    # Calculate EV assuming -110 odds (52.38% break-even)
                    breakeven = 0.5238
                    if prob > breakeven:
                        ev_percent = (prob - breakeven) / breakeven  # Rough EV calculation
                    else:
                        ev_percent = 0
                    
                    # Cap EV at reasonable levels
                    ev_percent = min(ev_percent, 0.5)  # Max 50% EV
                    
                    results.append({
                        'NAME': player_name,
                        'CATEGORY': 'player_points',
                        'LINE': line,
                        'SIDE': side,
                        'PREDICTION': prediction,
                        'OVER%': prob if side.upper().startswith('o') else 1-prob,
                        'UNDER%': 1-prob if side.upper().startswith('o') else prob,
                        'EV%': ev_percent,
                        'EV$': ev_percent * self.base_stake,
                        'KELLY QUARTER': max(0.01, ev_percent * 0.25),
                        'BOOKMAKER': 'prizepicks'
                    })
                    
                except Exception as e:
                    continue
            
            return pd.DataFrame(results)
            
        except Exception as e:
            return pd.DataFrame()
    
    def process_single_bets(self, game_date: str) -> List[BetResult]:
        """Process single bets with enhanced prediction"""
        results = []
        
        try:
            predictions = self.enhanced_single_bet(game_date)
            
            if predictions.empty:
                return results
            
            # Ensure EV% column exists and is numeric
            if 'EV%' not in predictions.columns:
                return results
                
            predictions['EV%'] = pd.to_numeric(predictions['EV%'], errors='coerce').fillna(0)
            
            # Filter by EV threshold and limit bets per day
            qualified_bets = predictions[predictions['EV%'] >= self.min_ev_threshold].sort_values('EV%', ascending=False)
            selected_bets = qualified_bets.head(self.max_bets_per_day)
            
            for _, bet in selected_bets.iterrows():
                try:
                    # Get actual result
                    actual = self.get_actual_result(bet['NAME'], game_date, 'PTS')
                    if actual is None:
                        continue
                    
                    # Safely get bet values with defaults
                    line = float(bet.get('LINE', 0))
                    side = str(bet.get('SIDE', 'over')).lower()
                    prediction = float(bet.get('PREDICTION', 0))
                    ev_percent = float(bet.get('EV%', 0))
                    
                    # Determine if bet won
                    if side.startswith('o'):
                        won = actual > line
                        probability = float(bet.get('OVER%', 0.5))
                    else:
                        won = actual < line
                        probability = float(bet.get('UNDER%', 0.5))
                    
                    # Calculate stake using Kelly criterion
                    kelly_quarter = float(bet.get('KELLY QUARTER', 0.01))
                    kelly_stake = self.base_stake * kelly_quarter * self.kelly_multiplier
                    kelly_stake = max(kelly_stake, self.base_stake * 0.01)  # Minimum 1% of base stake
                    
                    # Calculate profit/loss (PrizePicks typically pays 2:1)
                    if won:
                        profit = kelly_stake
                    else:
                        profit = -kelly_stake
                    
                    result = BetResult(
                        date=game_date,
                        bet_type='single',
                        player_name=bet['NAME'],
                        category=bet.get('CATEGORY', 'points'),
                        line=line,
                        side=side,
                        prediction=prediction,
                        actual_result=actual,
                        probability=probability,
                        ev_percent=ev_percent,
                        ev_dollars=float(bet.get('EV$', 0)),
                        kelly_fraction=kelly_quarter,
                        stake_amount=kelly_stake,
                        won=won,
                        profit_loss=profit,
                        bookmaker='prizepicks'
                    )
                    results.append(result)
                    
                except Exception as e:
                    continue
                    
        except Exception as e:
            pass
            
        return results
    
    def run_backtest(self, bet_types: List[str] = ['single']) -> None:
        """
        Run the backtest silently
        """
        dates = self.get_unique_dates()
        
        for i, date in enumerate(dates):
            daily_results = []
            daily_errors = []
            
            # Process single bets
            if 'single' in bet_types:
                try:
                    single_results = self.process_single_bets(date)
                    daily_results.extend(single_results)
                except Exception as e:
                    daily_errors.append(f"Single bets error: {e}")
            
            # Log the day's processing
            self.processing_log.append({
                'date': date,
                'bets_found': len(daily_results),
                'errors': daily_errors,
                'success': len(daily_results) > 0
            })
            
            # Store results
            if daily_results:
                self.bet_results.extend(daily_results)
                
                # Create daily summary
                daily_profit = sum(r.profit_loss for r in daily_results)
                daily_stake = sum(r.stake_amount for r in daily_results)
                wins = sum(1 for r in daily_results if r.won)
                
                self.daily_summary.append({
                    'date': date,
                    'num_bets': len(daily_results),
                    'total_stake': daily_stake,
                    'total_profit': daily_profit,
                    'wins': wins,
                    'hit_rate': wins / len(daily_results) if daily_results else 0,
                    'roi': daily_profit / daily_stake if daily_stake > 0 else 0
                })
    
    def calculate_performance_metrics(self) -> Dict[str, float]:
        """Calculate the exact metrics from your notes"""
        if not self.bet_results:
            return {}
        
        # Basic calculations
        total_bets = len(self.bet_results)
        winning_bets = sum(1 for r in self.bet_results if r.won)
        total_profit = sum(r.profit_loss for r in self.bet_results)
        total_amount_staked = sum(r.stake_amount for r in self.bet_results)
        
        # 1. ROI (%) = (Total profit ÷ Total amount staked) × 100
        roi_percent = (total_profit / total_amount_staked) * 100 if total_amount_staked > 0 else 0
        
        # 2. Hit rate = (Number of winning bets) ÷ (Total bets)
        hit_rate = winning_bets / total_bets if total_bets > 0 else 0
        
        # 3. Volatility = Standard deviation of daily P&L
        daily_pnl = [summary['total_profit'] for summary in self.daily_summary]
        volatility = np.std(daily_pnl, ddof=1) if len(daily_pnl) > 1 else 0
        
        # 4. Max drawdown = Largest peak-to-trough loss
        cumulative_profits = np.cumsum([r.profit_loss for r in self.bet_results])
        running_max = np.maximum.accumulate(cumulative_profits)
        drawdowns = running_max - cumulative_profits
        max_drawdown = np.max(drawdowns) if len(drawdowns) > 0 else 0
        
        # 5. Sharpe ratio = (Average daily return ÷ Std. dev. of daily returns) × √252
        if len(daily_pnl) > 1 and volatility > 0:
            avg_daily_return = np.mean(daily_pnl)
            sharpe_ratio = (avg_daily_return / volatility) * np.sqrt(252)
        else:
            sharpe_ratio = 0
        
        return {
            'total_profit': total_profit,
            'total_amount_staked': total_amount_staked,
            'roi_percent': roi_percent,
            'number_of_winning_bets': winning_bets,
            'total_bets': total_bets,
            'hit_rate': hit_rate,
            'volatility': volatility,
            'max_drawdown': max_drawdown,
            'average_daily_return': np.mean(daily_pnl) if daily_pnl else 0,
            'std_dev_daily_returns': volatility,
            'sharpe_ratio': sharpe_ratio,
            'total_days': len(self.daily_summary)
        }
    
    def print_summary(self) -> None:
        """Print summary showing exactly your specified metrics"""
        metrics = self.calculate_performance_metrics()
        
        print("\n" + "="*60)
        print("NBA PROPS BACKTEST - KEY METRICS")
        print("="*60)
        
        print(f"\nYOUR SPECIFIED METRICS:")
        print("-" * 40)
        
        # 1. ROI (%) = (Total profit ÷ Total amount staked) × 100
        print(f"ROI (%): {metrics['roi_percent']:.2f}%")
        print(f"  └─ Total profit: ${metrics['total_profit']:,.2f}")
        print(f"  └─ Total amount staked: ${metrics['total_amount_staked']:,.2f}")
        
        # 2. Hit rate = (Number of winning bets) ÷ (Total bets)
        print(f"\nHit rate: {metrics['hit_rate']:.4f} ({metrics['hit_rate']:.2%})")
        print(f"  └─ Number of winning bets: {metrics['number_of_winning_bets']:,}")
        print(f"  └─ Total bets: {metrics['total_bets']:,}")
        
        # 3. Volatility = Standard deviation of daily P&L
        print(f"\nVolatility: ${metrics['volatility']:.2f}")
        print(f"  └─ Standard deviation of daily P&L")
        print(f"  └─ Based on {metrics['total_days']} trading days")
        
        # 4. Max drawdown = Largest peak-to-trough loss
        print(f"\nMax drawdown: ${metrics['max_drawdown']:.2f}")
        print(f"  └─ Largest peak-to-trough loss")
        
        # 5. Sharpe ratio = (Average daily return ÷ Std. dev. of daily returns) × √252
        print(f"\nSharpe ratio: {metrics['sharpe_ratio']:.4f}")
        print(f"  └─ Average daily return: ${metrics['average_daily_return']:.2f}")
        print(f"  └─ Std. dev. of daily returns: ${metrics['std_dev_daily_returns']:.2f}")
        print(f"  └─ Annualized (× √252): {metrics['sharpe_ratio']:.4f}")

def run_clean_backtest():
    # Initialize backtester
    backtester = CleanNBABacktester(
        min_ev_threshold=0.05,  # 2% minimum EV
        base_stake=100,
        kelly_multiplier=0.15,
        max_bets_per_day=3,
        start_date='2024-10-22',
        end_date='2025-04-12'
    )
    
    # Run backtest silently
    backtester.run_backtest(bet_types=['single'])
    
    # Print summary
    backtester.print_summary()
    
    return backtester

if __name__ == "__main__":
    # Run the clean backtest
    backtest_results = run_clean_backtest()


NBA PROPS BACKTEST - KEY METRICS

YOUR SPECIFIED METRICS:
----------------------------------------
ROI (%): 20.33%
  └─ Total profit: $127.57
  └─ Total amount staked: $627.54

Hit rate: 0.6012 (60.12%)
  └─ Number of winning bets: 202
  └─ Total bets: 336

Volatility: $3.37
  └─ Standard deviation of daily P&L
  └─ Based on 133 trading days

Max drawdown: $48.99
  └─ Largest peak-to-trough loss

Sharpe ratio: 4.5240
  └─ Average daily return: $0.96
  └─ Std. dev. of daily returns: $3.37
  └─ Annualized (× √252): 4.5240
